<a href="https://colab.research.google.com/github/Rak3shb2003/Rakeshbabu_INFO5731_Spring2026-/blob/main/In_Class_Exercise_Combined_8%269.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# In-Class Exercise: Sentiment Analysis + Text Classification + Prompt Engineering

**Course:** NLP / Text Mining  
**Total Points:** 30  
**Suggested Time:** about 45 minutes total  
**Dataset:** `sentiment_classification_combined_data.csv`

**Instructions**
- Complete all questions in this notebook.
- Run all cells before submission.
- Keep your answers clear and organized.
- For the prompt engineering part, you do **not** need an API key.


## Part A — Sentiment Analysis with VADER (10 points)
In this part, use **VADER** to compute sentiment scores for the text data.

In [1]:
# Upload the dataset file from your computer (Google Colab)
from google.colab import files

uploaded = files.upload()
filename = next(iter(uploaded))
print("Uploaded file name:", filename)


Saving sentiment_classification_combined_data.csv to sentiment_classification_combined_data.csv
Uploaded file name: sentiment_classification_combined_data.csv


### Q1. Load and inspect the dataset (2 points)
Show:
1. the first 5 rows  
2. the shape of the dataset  
3. the column names

In [5]:
# Read the uploaded CSV file
import pandas as pd

df = pd.read_csv(filename)

# Show the first 5 rows
print("First 5 rows:")
print(df.head())

# Show the shape of the dataset
print("\nShape of the dataset (rows, columns):")
print(df.shape)

# Show the column names
print("\nColumn names:")
print(df.columns)

First 5 rows:
                                                Text TrueLabel  Score
0  I loved this healthcare AI workshop. The examp...  positive      5
1  This lecture was okay, but some parts were con...   neutral      3
2  The notebook was easy to follow and very pract...  positive      5
3  I did not like the explanation of the model re...  negative      1
4  The activity was interesting and helped me und...  positive      4

Shape of the dataset (rows, columns):
(24, 3)

Column names:
Index(['Text', 'TrueLabel', 'Score'], dtype='object')


### Q2. Apply VADER to compute sentiment scores (3 points)
- Import VADER
- Download the lexicon if needed
- Create a new column called `compound`
- Show the first 5 rows of `Text` and `compound`

In [3]:
!pip -q install nltk
import nltk
nltk.download('vader_lexicon')
from nltk.sentiment import SentimentIntensityAnalyzer
sia = SentimentIntensityAnalyzer()

[nltk_data] Downloading package vader_lexicon to /root/nltk_data...


In [4]:
# Apply VADER sentiment scoring
df['compound'] = df['Text'].apply(lambda x: sia.polarity_scores(str(x))['compound'])

# Show first 5 rows with Text and compound score
df[['Text', 'compound']].head()

,Text,compound
0,I loved this healthcare AI workshop. The examp...,0.8555
1,"This lecture was okay, but some parts were con...",-0.2263
2,The notebook was easy to follow and very pract...,0.4404
3,I did not like the explanation of the model re...,-0.2755
4,The activity was interesting and helped me und...,0.4019


### Q3. Create a sentiment label from the compound score (3 points)
Create a new column called `PredictedSentiment` using this rule:
- `compound >= 0.05` → `positive`
- `compound <= -0.05` → `negative`
- otherwise → `neutral`

Then show the first 10 rows of `Text`, `compound`, and `PredictedSentiment`.

In [13]:
import pandas as pd
from nltk.sentiment import SentimentIntensityAnalyzer
import nltk

# Download VADER lexicon (run once)
nltk.download('vader_lexicon')

# Load dataset (make sure filename is defined)
df = pd.read_csv(filename)

# Initialize sentiment analyzer
sia = SentimentIntensityAnalyzer()

# STEP 1: Create compound score column
# CHANGE "Text" to "text" if your dataset uses lowercase column name
df["compound"] = df["Text"].apply(lambda x: sia.polarity_scores(str(x))["compound"])

# STEP 2: Create sentiment label function
def get_sentiment(compound):
    if compound >= 0.05:
        return "positive"
    elif compound <= -0.05:
        return "negative"
    else:
        return "neutral"

# STEP 3: Create PredictedSentiment column
df["PredictedSentiment"] = df["compound"].apply(get_sentiment)

# STEP 4: Show required output
print(df[["Text", "compound", "PredictedSentiment"]].head(10))

                                                Text  compound  \
0  I loved this healthcare AI workshop. The examp...    0.8555   
1  This lecture was okay, but some parts were con...   -0.2263   
2  The notebook was easy to follow and very pract...    0.4404   
3  I did not like the explanation of the model re...   -0.2755   
4  The activity was interesting and helped me und...    0.4019   
5  The class was too fast and I could not follow ...    0.0000   
6  The topic is important, but the instructions w...    0.1027   
7  I really enjoyed the examples and the live cod...    0.5563   
8  The dataset was messy and the task felt frustr...   -0.6597   
9  The exercise was manageable and useful for pra...    0.4404   

  PredictedSentiment  
0           positive  
1           negative  
2           positive  
3           negative  
4           positive  
5            neutral  
6           positive  
7           positive  
8           negative  
9           positive  


[nltk_data] Downloading package vader_lexicon to /root/nltk_data...
[nltk_data]   Package vader_lexicon is already up-to-date!


### Q4. Compare review score and sentiment (2 points)
Group by `Score` and compute the **average compound score** for each score value.

In [14]:
# Group by Score and calculate average compound sentiment
score_sentiment = df.groupby("Score")["compound"].mean().reset_index()

# Rename column for clarity (optional but good for submission)
score_sentiment.columns = ["Score", "Average_Compound"]

# Display result
print(score_sentiment)

   Score  Average_Compound
0      1         -0.507033
1      2         -0.296125
2      3         -0.150783
3      4          0.478340
4      5          0.621167


## Part B — Prompt Engineering (10 points)
This part is based on the **Prompt Engineering** notebook. You do **not** need an API key. Focus on writing better prompts using the concepts of **instruction, context, output format, zero-shot, few-shot, and chain-of-thought**.

In [15]:
# Sample texts for the prompt engineering questions
sample_reviews = [
    "The product arrived late and the quality was poor.",
    "Amazing battery life and elegant design.",
    "It is fine for daily use, nothing special."
]
for i, text in enumerate(sample_reviews, start=1):
    print(f"{i}. {text}")

1. The product arrived late and the quality was poor.
2. Amazing battery life and elegant design.
3. It is fine for daily use, nothing special.


### Q1. Zero-shot classification prompt (3 points)
Write a **zero-shot prompt** that asks an AI model to classify each review as **positive, negative, or neutral**.

Your prompt should include:
- a clear instruction
- the three labels
- a request for a clean output format

Store your prompt in a Python variable called `zero_shot_prompt`, then print it.

In [16]:
zero_shot_prompt = """
You are a sentiment classification model.

Task:
Classify each review into one of the following categories:
- positive
- negative
- neutral

Instructions:
- Read the review carefully.
- Assign only ONE label per review.
- Do not provide explanations.

Output format:
Return results in this format only:
Review: <text>
Sentiment: <positive/negative/neutral>
"""

print(zero_shot_prompt)


You are a sentiment classification model.

Task:
Classify each review into one of the following categories:
- positive
- negative
- neutral

Instructions:
- Read the review carefully.
- Assign only ONE label per review.
- Do not provide explanations.

Output format:
Return results in this format only:
Review: <text>
Sentiment: <positive/negative/neutral>



### Q2. Few-shot prompt improvement (3 points)
Now improve the previous prompt by adding **two examples**.

Requirements:
- Store it in a variable called `few_shot_prompt`
- Include 2 input-output examples
- Ask the model to classify the 3 sample reviews after the examples

In [17]:
few_shot_prompt = """
You are a sentiment classification model.

Task:
Classify each review into one of the following categories:
- positive
- negative
- neutral

Instructions:
- Read the review carefully.
- Assign only ONE label per review.
- Do not provide explanations.

Examples:

Review: "The movie was amazing and I loved every part of it."
Sentiment: positive

Review: "The product broke after one day and I am very disappointed."
Sentiment: negative

Now classify the following reviews:

Review: "The service was okay, nothing special but not bad either."
Sentiment:

Review: "I absolutely love this phone, it works perfectly and is fast."
Sentiment:

Review: "The food was cold and the delivery was late."
Sentiment:
"""

print(few_shot_prompt)


You are a sentiment classification model.

Task:
Classify each review into one of the following categories:
- positive
- negative
- neutral

Instructions:
- Read the review carefully.
- Assign only ONE label per review.
- Do not provide explanations.

Examples:

Review: "The movie was amazing and I loved every part of it."
Sentiment: positive

Review: "The product broke after one day and I am very disappointed."
Sentiment: negative

Now classify the following reviews:

Review: "The service was okay, nothing special but not bad either."
Sentiment:

Review: "I absolutely love this phone, it works perfectly and is fast."
Sentiment:

Review: "The food was cold and the delivery was late."
Sentiment:



### Q3. Prompt comparison and refinement (4 points)
Create a short table or DataFrame with **three columns**:
- `PromptType`
- `MainFeature`
- `WhyUseful`

Include these three prompt types:
1. Zero-shot
2. Few-shot
3. Chain-of-thought

Then, in **2–3 sentences**, explain which one you would choose for a more difficult text analysis task and why.

In [18]:
import pandas as pd

data = {
    "PromptType": ["Zero-shot", "Few-shot", "Chain-of-thought"],
    "MainFeature": [
        "No examples provided, only instructions",
        "Includes a few labeled examples to guide output",
        "Encourages step-by-step reasoning before final answer"
    ],
    "WhyUseful": [
        "Fast and simple, works well for straightforward tasks",
        "Improves accuracy by showing expected patterns",
        "Better for complex tasks requiring reasoning and justification"
    ]
}

df_prompts = pd.DataFrame(data)

print(df_prompts)

         PromptType                                        MainFeature  \
0         Zero-shot            No examples provided, only instructions   
1          Few-shot    Includes a few labeled examples to guide output   
2  Chain-of-thought  Encourages step-by-step reasoning before final...   

                                           WhyUseful  
0  Fast and simple, works well for straightforwar...  
1     Improves accuracy by showing expected patterns  
2  Better for complex tasks requiring reasoning a...  


For more difficult text analysis tasks, I would choose chain-of-thought prompting because it encourages the model to reason step by step instead of guessing directly. This improves accuracy when the task involves ambiguity, multiple cues, or deeper interpretation of text. Few-shot prompting is also helpful, but chain-of-thought is more effective when reasoning quality matters more than pattern matching.

## Reflection (optional, no extra points)
Briefly explain the difference between:
- sentiment analysis
- text classification
- prompt engineering

Sentiment analysis is a specific type of NLP task that focuses on identifying the emotional tone of text, usually as positive, negative, or neutral. For example, it determines whether a movie review expresses happiness, frustration, or indifference.

Text classification is a broader machine learning task where text is assigned to any predefined category, not just sentiment. This can include topics like spam vs. not spam, sports vs. politics, or customer complaint categories.

Prompt engineering is the practice of designing and refining inputs (prompts) given to a language model to get better, more accurate, or more structured outputs. Instead of changing the model, you improve how you ask the question to guide the model’s response effectively.